___
## Modify a table with sensitive data 

### Step 1: Create a backup table

In [1]:
import sqlite3
from datetime import datetime

def create_table_backup(table_name, backup_sufix=None):
    """ 
    This function creates timestamped backup of any table for safety

    parameters: 
    table_name(str): Name of the table to backup
    backup_sufix(str): Custom suffix for backup table name 
    """
    conn = None
    # The main operation that might fail
    try:
        conn = sqlite3.connect('nursery.db')
        cursor = conn.cursor()

        # Generate the backup table with timestamp
        if backup_sufix:
            backup_table = f"{table_name}_BACKUP_{backup_sufix}"
        else:
            timestamp = datetime.now().strftime("%m_%d_%y")
            backup_table = f"{table_name}_BACKUP_{timestamp}"
        
        # Execute backup
        cursor.execute(f"""
        CREATE TABLE {backup_table} AS 
        SELECT * FROM {table_name}
        """)

        # Verify backup was created
        cursor.execute(f"SELECT COUNT(*) FROM {backup_table}")
        record_count = cursor.fetchone()[0]

        conn.commit()

        print(f"☑️ Backup successful: {backup_table}")
        print(f"📊 Records backed up: {record_count}")

        return backup_table

    # Handle database errors specifically
    except sqlite3.Error as e:  
        print (f"🚫 Backup failed for {table_name}: {e}")
        if conn:
            conn.rollback()
        return None
    
    # Ensure connection always closes even if error occurs
    finally:
        if conn:
            conn.close()


# --- PRACTICE USAGE ---
if __name__ == "__main__":
    # basic usage
    backup_name = create_table_backup("OBSERVACION_LOTE")


☑️ Backup successful: OBSERVACION_LOTE_BACKUP_12_16_25
📊 Records backed up: 970


### Step 2: Modofy the date format

SQLite native support: SQLite handles yyyy-mm-dd format natively for date operations

- Chronological sorting: Dates sort correctly as text without conversion
- ISO 8601 standard: Widely accepted international standard

In [8]:
import sqlite3

def convert_to_iso_date(table_name, date_column='FECHA_OBSERVACION'):
    """ 
    Convert date format from mm-dd-yy to yyyy-mm-dd (ISO 8601) for SQLite compatibility

    Parameters
    table_name: Name of the table to update
    date_column: Name of the date field
    """

    conn = None
    try:
        conn = sqlite3.connect('nursery.db')
        cursor = conn.cursor()

        # SQL to update the date format from mm-dd-yy to yyyy-mm-dd
        # Extract month,day, year and prepend 20 to year
        update_query = f"""
            UPDATE {table_name}
            SET {date_column} =
                '20' || substr({date_column}, 7, 2) || '-' ||
                substr({date_column}, 1 ,2) || '-' ||
                substr({date_column}, 4, 2)
            WHERE {date_column} LIKE '__-__-__'
        """

        cursor.execute(update_query)
        rows_updated = cursor.rowcount

        conn.commit()

        print(f"✅ Date format converted to ISO in {table_name}, {date_column}")
        print(f"📊 Rows updated: {rows_updated}")

        return rows_updated

    except sqlite3.Error as e:
        print(f"Error converting the dates: {e}")
        if conn:
            conn.rollback()
        return None

    finally:
        if conn:
            conn.close()

# ----USAGE----
if __name__ == "__main__":
    convert_to_iso_date('OBSERVACION_LOTE')

✅ Date format converted to ISO in OBSERVACION_LOTE, FECHA_OBSERVACION
📊 Rows updated: 970


### Step 3: update field values

In [13]:
import sqlite3

def update_field_values(table_name, field_name, value_mappings):
    """ 
    Update specific valuesin a table field using case-insensitive matching.
    For data cleaning, category standarization and bulk field updates.

    Parameters:
    table_name (str): Table to update
    field_name (str): Column containing values to change
    value_mappings (dict): Dictionary of {old_value: new_value} pairs
    """

    conn = None
    try:
        conn = sqlite3.connect('nursery.db')
        cursor = conn.cursor()

        # Build a dynamic case statement for multiple value changes
        case_statements = []
        for old_val, new_val in value_mappings.items():
            case_statements.append(f"WHEN UPPER({field_name}) = UPPER('{old_val}') THEN '{new_val}'")

            case_sql = "\n                ".join(case_statements)

            update_query = f"""
                UPDATE {table_name}
                SET {field_name} = CASE 
                    {case_sql}
                    ELSE {field_name}
                END
                WHERE {field_name} IS NOT NULL
            """

            cursor.execute(update_query)
            rows_updated = cursor.rowcount

            conn.commit()

            print(f"👌🏽 Updated {field_name} in {table_name}")
            print(f"📊 Rows affected: {rows_updated}")
            print(f"♻️ Changes applied: {len(value_mappings)} value mappings")

            return rows_updated
        
    except sqlite3.Error as e:
        print(f"Error updating {field_name}: {e}")
        if conn:
            conn.rollback()
        return None
    
    finally:
        if conn:
            conn.close()

# --- PRACTICE ---
if __name__ == "__main__":
    # single category change
    update_field_values(
        'OBSERVACION_LOTE_BACKUP_10_31_25',
        'ETAPA_CRECIMIENTO',
        {'CRECIMIENTO VEGETATIVO': 'BOLSA_PEQUEÑA'
         # add as many category changes as needed
         }
    )

👌🏽 Updated ETAPA_CRECIMIENTO in OBSERVACION_LOTE_BACKUP_10_31_25
📊 Rows affected: 176
♻️ Changes applied: 1 value mappings


### Final step: migrate the backup data to the new schema

WARNING! RUN THIS AFTER DROPPING THE ORIGINAL TABLE AND REDEFINING THE TABLE STRUCTURE IF NEEDED

In [6]:
import sqlite3

def migrate_backup_to_main(backup_table_name, main_table_name="REGISTRO_RIEGO"):
    """
    Copies all data from backup table to main table, then drops the backup.
    Use when you have a new schema table ready and want to migrate historical data.
    
    Parameters:
    backup_table_name (str): Name of the backup table with historical data
    main_table_name (str): Name of the main table with new schema
    """
    conn = None
    try:
        conn = sqlite3.connect('nursery.db')
        cursor = conn.cursor()
        
        # Step 1: Copy all data from backup to main table
        copy_query = f"""
        INSERT INTO {main_table_name} 
        (ESPECIE_LOTE_ID, FECHA, CANTIDAD_AGUA_LT, NOTAS)
        SELECT 
            ESPECIE_LOTE_ID, FECHA, CANTIDAD_AGUA_LT, NOTAS
        FROM {backup_table_name}
        """
        
        cursor.execute(copy_query)
        records_copied = cursor.rowcount
        
        # Step 2: Drop the backup table
        cursor.execute(f"DROP TABLE {backup_table_name}")
        
        conn.commit()
        
        print(f"✅ Successfully migrated {records_copied} records")
        print(f"✅ Backup table '{backup_table_name}' dropped")
        print("🎉 Historical data preserved and ready for new!")
        
        return records_copied
        
    except sqlite3.Error as e:
        print(f"❌ Error during migration: {e}")
        if conn:
            conn.rollback()
        return None
    
    finally:
        if conn:
            conn.close()

# --- FINAL MIGRATION ---
if __name__ == "__main__":
    migrate_backup_to_main("REGISTRO_RIEGO_BACKUP_11_10_25")

✅ Successfully migrated 239 records
✅ Backup table 'REGISTRO_RIEGO_BACKUP_11_10_25' dropped
🎉 Historical data preserved and ready for new!
